# 08 · Healthy-only autoencoder (Q1) on cache C

## Purpose
Q1 asks whether the machine is healthy, using a detector trained **only on healthy recordings**. This notebook evaluates
autoencoders on the frozen representation C and compares them, fold by fold and on identical recordings, with the best
classical detector (B1s, the speed-aware polynomial baseline: per-fold AUROC 0.990). The autoencoder may not beat it; its
value (an alternative detector and sensor anomaly evidence) has to be shown by the results.

## Protocol (identical to the corrected Q1 protocol of Stage 2D.1)
- Outer dev folds k = 0, 1, 2 (`results/dev_folds.csv`). For each k, the healthy recordings of the other folds are split
  by the **saved** Stage 2D.1 assignment into **fit** (11 per fold) and **calibration** (6, 6, 5).
- Normalisation: `norm_stats` in development mode, fit_ids = the fit normals only.
- Training: fit normals only, a fixed number of epochs from the config, one seed (42), no early stopping.
- Threshold: the maximum calibration score of the same fitted model (higher = more anomalous; ties do not alarm).
- Evaluation: fold k only, every recording once: its healthy recordings (25 in total over the folds) and all faults.

## The models
- **Primary: frame-level autoencoder (AE-7 with microphone, AE-6 vibration only).** A deliberate change from the plan's
  window-level model: each time frame (7 × 128 = channels × spectral bins; vibration bins are orders, microphone bins
  are Mel frequencies) is one example, encoded by 1D convolutions over the bin axis into a 16-d bottleneck. It
  reconstructs **individual spectral slices and does not model temporal relationships across frames**. Frames give more
  training examples (99 windows × 25 frames per fold) but come from only 11 healthy recordings per fold: they are not
  independent recordings, and all frames and windows of a recording stay together in every split.
- **Secondary: window-level autoencoder (WAE-7, WAE-6)** on 7 × 128 × 25 with a 64-d bottleneck, as in the plan.
- **Reconstruction error (defined before training):** squared error in units normalised with the healthy fit recordings;
  per-channel error = mean over that channel's bins and frames; total = equal-weight mean over channels; a recording's
  score = mean over all frames of its 9 windows. Training uses gain augmentation (≤ 1 dB) and the model reconstructs the
  **same augmented input** it receives; evaluation uses no augmentation.

**Caveat:** 25 held-out healthy recordings from **one folder**; B1s's per-fold AUROC of 0.990 leaves little room.
All numbers are **development findings**, not reserved-test results.

## Inputs
`results/ae/<variant>_s42_dev_eval.json`, `results/ae/ae_vs_b1s_s42.json`, `results/ae/sensor_evidence_*.csv`,
`results/scores/<variant>_s42_dev_channel_errors.parquet`, `results/scores/b1s_*_s42_dev_scores.parquet` and the MLflow
runs (`stage = 3B`, `family = ae`), written by `scripts/08_train_ae.py` and `scripts/11b_b1s_scores.py`.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Loads the settings, <code>metadata.csv</code>, the autoencoder and B1s results and the per-recording scores, and stops if any result was produced under different split assignments.</div>

In [ ]:
import json

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd

from ml.cache import check_split_hash
from ml.config import load_config
from ml.paths import DEFAULT_CONFIG, SYSTEM_CODE_DIR, RunPaths, mlflow_uri
from ml.q1 import FAULTS

cfg = load_config(DEFAULT_CONFIG)
meta = pd.read_csv(SYSTEM_CODE_DIR / "metadata.csv")
FIG = SYSTEM_CODE_DIR / "results" / "figures"
RES = SYSTEM_CODE_DIR / "results" / "ae"
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
AE = ["ae7", "ae6", "wae7", "wae6"]
LABEL = {"ae7": "AE-7 (frame, +mic)", "ae6": "AE-6 (frame, vib.)", "wae7": "WAE-7 (window, +mic)",
         "wae6": "WAE-6 (window, vib.)", "b1s_all": "B1s (all features)", "b1s_nomic": "B1s (no mic)"}
ev = {v: json.loads(RunPaths("ae", v, 42).result("eval").read_text()) for v in AE}
cmp_ = json.loads((RES / "ae_vs_b1s_s42.json").read_text())
check_split_hash(cmp_["provenance"]["split_hash"], meta, "ae_vs_b1s_s42.json")
for v, e in ev.items():
    check_split_hash(e["provenance"]["split_hash"], meta, f"{v} eval")
scores = {v: pd.read_parquet(RunPaths("ae", v, 42).scores("channel_errors")) for v in AE}
scores.update({b: pd.read_parquet(RunPaths("q1q3", b, 42).scores()) for b in ("b1s_all", "b1s_nomic")})
det = cmp_["detectors"]
print(cmp_["provenance"]["basis"])
print(cmp_["provenance"]["healthy_caveat"])

## 1. Architectures and fixed settings

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Builds each autoencoder, counts its trainable parameters, and lists the fixed settings from the config (including the reconstruction target and the number of epochs).</div>

In [ ]:
from ml.models.ae import build_ae
from ml.models.cnn import n_parameters

a = cfg.ae
display(pd.DataFrame([{"variant": LABEL[v], "level": lv, "input": f"{ch} × 128" + (" × 25" if lv == "window" else ""),
                       "bottleneck": a.bottleneck if lv == "frame" else a.w_bottleneck,
                       "parameters": n_parameters(build_ae(lv, ch, a)),
                       "epochs": a.epochs if lv == "frame" else a.w_epochs,
                       "batch": a.batch_size if lv == "frame" else a.w_batch_size,
                       "train s (3 folds, MPS)": round(ev[v]["train_seconds_total"], 1)}
                      for v, lv, ch in (("ae7", "frame", 7), ("ae6", "frame", 6), ("wae7", "window", 7),
                                        ("wae6", "window", 6))]).set_index("variant"))
pd.Series({"frame AE": f"Conv1d {a.conv_channels} (kernel {a.kernel}, stride 2) → Linear → z ({a.bottleneck}) → mirrored",
           "window AE": f"Conv2d {a.w_conv_channels} (3×3, stride 2) → Linear → z ({a.w_bottleneck}) → mirrored",
           "optimiser": f"AdamW lr {a.lr}, weight decay {a.weight_decay}; {a.warmup_epochs} warm-up epochs, cosine",
           "augmentation": f"gain ±{a.gain_jitter_db} dB per example and channel (training only)",
           "reconstruction target": a.reconstruct, "seed": a.seed}, name="setting").to_frame()

## 2. Training curves
Training reconstruction loss (normalised units, augmented fit examples) per fold, from MLflow.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Reads each fold's per-epoch training loss from MLflow and plots it for every autoencoder variant, one line per fold, on a logarithmic axis.</div>

In [ ]:
mlflow.set_tracking_uri(mlflow_uri(cfg))
client = mlflow.tracking.MlflowClient()
runs = mlflow.search_runs(experiment_names=["machinesense"], filter_string="tags.stage = '3B' and tags.family = 'ae'")
fig, axes = plt.subplots(1, 4, figsize=(15, 3.4), sharey=True)
for ax, v in zip(axes, AE):
    kids = runs[runs["tags.mlflow.parentRunId"] == ev[v]["provenance"]["mlflow_run_id"]].sort_values("tags.fold")
    for _, r in kids.iterrows():
        h = client.get_metric_history(r.run_id, "train_loss")
        ax.plot([m.step for m in h], [m.value for m in h], label=f"fold {r['tags.fold']}")
    ax.set(yscale="log", xlabel="epoch", title=LABEL[v])
axes[0].set_ylabel("training MSE (normalised units)")
axes[0].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG / "08_ae_training_curves.png")

## 3. Error distributions per fold (AE-7, primary)
Each row is one outer fold's model; each panel compares that fold's held-out healthy recordings with one fault class.
The dashed line is the fold's threshold (maximum calibration score). Scores are on a log axis.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> For AE-7, draws one histogram panel per fold and fault class: held-out healthy recordings (grey) against that fault class (colour), with the fold's calibrated threshold as a dashed line. Separation means the fault class sits to the right of the healthy recordings.</div>

In [ ]:
s = scores["ae7"]
t = s[s.role == "test"]
fig, axes = plt.subplots(3, len(FAULTS), figsize=(22, 7), sharey="row")
bins = np.logspace(np.log10(t.score.min() * 0.9), np.log10(t.score.max() * 1.1), 30)
for k in range(3):
    g = t[t.outer_fold == k]
    thr = g.threshold.iloc[0]
    for j, c in enumerate(FAULTS):
        ax = axes[k, j]
        ax.hist(g[g.class10 == "normal"].score, bins=bins, color="grey", alpha=0.7, label="healthy")
        ax.hist(g[g.class10 == c].score, bins=bins, color=plt.cm.tab10([0, 1, 2, 3, 4, 5, 6, 8, 9][j]), alpha=0.6, label=c)
        ax.axvline(thr, color="k", ls="--", lw=0.8)
        ax.set_xscale("log")
        if k == 0:
            ax.set_title(c.replace("_misalignment", "_mis"), fontsize=8)
        if j == 0:
            ax.set_ylabel(f"fold {k}: recordings")
        if k == 2:
            ax.set_xlabel("recording error", fontsize=8)
fig.suptitle("AE-7 recording reconstruction error: held-out healthy (grey) vs each fault class, per fold model")
fig.tight_layout()
fig.savefig(FIG / "08_ae_error_histograms.png")

## 4. Thresholds, false alarms and acceptance

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Tabulates per detector and fold: fit and calibration counts, the threshold (maximum calibration score), false alarms among that fold's held-out healthy recordings and the measured acceptance; then the totals out of 25.</div>

In [ ]:
rows = []
for name, d in det.items():
    for f in d["per_fold"]:
        rows.append({"detector": LABEL[name], "fold": f["fold"], "fit": f["n_fit"], "calibration": f["n_cal"],
                     "threshold": round(f["threshold"], 4), "held-out healthy": f["n_heldout_normal"],
                     "false alarms": f["false_alarms"], "acceptance": round(f["measured_acceptance"], 3)})
display(pd.DataFrame(rows).set_index(["detector", "fold"]))
pd.DataFrame([{"detector": LABEL[n], "false alarms": f"{d['false_alarms_total']}/{d['heldout_normals_total']}",
               "approx. 95 % interval": f"[{d['false_alarm_ci95_approx'][0]:.2f}, {d['false_alarm_ci95_approx'][1]:.2f}]",
               "measured acceptance": round(d["measured_acceptance_total"], 2),
               "theoretical reference": [round(x, 3) for x in d["theoretical_acceptance_reference"]],
               "detection overall": round(d["detection_overall"], 3)} for n, d in det.items()]).set_index("detector")

## 5. Within-fold comparison with B1s (identical recordings)
The basis for conclusions: per-fold AUROC of each detector on the same fold-k recordings, and their per-fold differences
(each autoencoder against B1s on the same sensor set). The pooled raw-score AUROC is shown only as a supplement: different
fold models have different score scales.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows the per-fold overall AUROC of every detector with the fold mean and spread, then the per-fold differences (autoencoder − B1s, and with vs without microphone), then per-class fold-mean AUROC.</div>

In [ ]:
display(pd.DataFrame([{"detector": LABEL[n], **{f"fold {f['fold']}": round(f["auroc"]["overall"], 4) for f in d["per_fold"]},
                       "mean": round(d["auroc_fold_mean"]["overall"], 4), "std": round(d["auroc_fold_std"]["overall"], 4),
                       "pooled raw AUROC (suppl.)": round(cmp_["pooled_raw_score_auroc_supplementary"][n], 4)}
                      for n, d in det.items()]).set_index("detector"))
display(pd.DataFrame({k: v["per_fold_auroc_diff"]["overall"] for k, v in cmp_["differences"].items()},
                     index=["fold 0", "fold 1", "fold 2"]).T.round(4))
pd.DataFrame({LABEL[n]: d["auroc_fold_mean"] for n, d in det.items()}).round(3)

## 6. Detection at each detector's own threshold

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Draws detection rates by fault class and at the mildest setting of each family (imbalance 6 g, horizontal 0.5 mm, vertical 0.51 mm, bearing 0 g = no added imbalance mass), each detector at its own calibrated threshold.</div>

In [ ]:
show = ["ae7", "ae6", "wae7", "b1s_all", "b1s_nomic"]
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5))
for ax, key, title in ((axes[0], "detection_by_class", "detection by fault class"),
                       (axes[1], "detection_mildest", "detection at the mildest setting")):
    cats = list(det["ae7"][key])
    w = 0.8 / len(show)
    for i, n in enumerate(show):
        vals = [det[n][key][c] if key == "detection_by_class" else det[n][key][c]["detection"] for c in cats]
        ax.bar(np.arange(len(cats)) + (i - (len(show) - 1) / 2) * w, vals, width=w,
               label=f"{LABEL[n]}: FA {det[n]['false_alarms_total']}/25")
    ax.set_xticks(range(len(cats)), [c.replace("_misalignment", "_mis") for c in cats], rotation=35, ha="right", fontsize=8)
    ax.set(ylim=(0, 1.25), ylabel="detection rate at own threshold", title=title)
axes[0].legend(fontsize=7, ncol=3, loc="upper center")
fig.tight_layout()
fig.savefig(FIG / "08_ae_detection.png")
pd.DataFrame({LABEL[n]: {c: v["detection"] for c, v in det[n]["detection_mildest"].items()} for n in show}).round(3)

## 7. Sensor anomaly evidence
For each fault recording, the channel with the largest per-channel reconstruction error under AE-7 (its own fold model).
**The channel with the largest reconstruction error is the one the autoencoder reconstructs least well; it is not proof of
the fault's physical location, nor an explanation of the classifier's decision.** Channels differ in baseline error, so a
supplementary table divides each channel's error by the same fold model's calibration-healthy median for that channel.
CLAUDE.md §3 records an unresolved observation that bearing faults and misalignment respond most on the opposite-named
accelerometer group.

<div style="background-color:#e3f2fd; color:#0d2538; padding:10px 14px; border-radius:6px; border-left:4px solid #64b5f6; margin:4px 0;"><b>What this code does:</b> Shows two cross-tables for AE-7: fault class (rows) against the channel with the largest reconstruction error (columns), counting fault recordings; first in absolute error, then relative to healthy calibration error.</div>

In [ ]:
display(pd.read_csv(RES / "sensor_evidence_ae7_s42.csv", index_col=0))
pd.read_csv(RES / "sensor_evidence_relative_ae7_s42.csv", index_col=0)

## Findings
Development findings (3 folds, 25 held-out healthy recordings from one folder, one seed); not reserved-test results.
- **Separation matches B1s, it does not beat it.** Per-fold overall AUROC: AE-7 0.990 ± 0.006 (folds 0.989, 0.983,
  0.998), AE-6 0.991 ± 0.006, WAE-7 0.990 ± 0.009, B1s 0.990 ± 0.013 (0.971, 1.000, 0.998). The within-fold differences
  AE-7 − B1s are +0.018, −0.017, +0.000: they change sign, so there is no consistent winner.
- **The operating points differ.** At each detector's own threshold (maximum of 6/6/5 calibration healthy): AE-7 raised
  0/25 false alarms and detected 95.8 % of faults; B1s raised 5/25 and detected 100 %. The autoencoders miss mainly
  horizontal misalignment (AE-7 detects 63 %; 44 % at the mildest 0.5 mm) and some overhang outer race (95 %); B1s detects
  both fully. With 25 healthy recordings, 0/25 and 5/25 are not a reliable ranking (approximate 95 % intervals
  [0, 0.14] and [0.07, 0.41]).
- **Microphone:** AE-7 − AE-6 per-fold AUROC differences are −0.002, −0.001, +0.000: no measurable effect on separation.
- **Frame vs window level:** the window-level AE (64-d bottleneck) is similar (0.990 ± 0.009) and detects more horizontal
  misalignment (75 %) at 2/25 false alarms; the frame-level AE does not model temporal structure.
- **Sensor anomaly evidence (AE-7):** for most classes the least well reconstructed channel is on the **overhang**
  accelerometer, including all underhang bearing classes (underhang outer → over_radial 72/73; underhang ball →
  over_tangential 75/77), consistent with the unresolved opposite-group observation of CLAUDE.md §3. Imbalance and
  vertical misalignment → over_axial. This shows which channel the autoencoder reconstructs least well, not where the
  fault physically is.
- **Cost:** about 7–11 s of MPS training per fold (98k-parameter frame AE; 294k window AE).


## Outputs written
- `results/figures/08_ae_training_curves.png`, `08_ae_error_histograms.png`, `08_ae_detection.png`
- (by `scripts/08_train_ae.py`) `results/ae/*`; `results/scores/<variant>_s42_dev_channel_errors.parquet`;
  `artifacts/ae/<variant>_s42_dev/fold<k>.pt`; MLflow runs tagged `stage = 3B`, `family = ae`.